In [ ]:
!apt-get update -y
!apt-get install -y ffmpeg
!pip install -q openai-whisper librosa soundfile tqdm

import os
import zipfile
import shutil
from tqdm import tqdm
import numpy as np
import whisper
import librosa
import re
from google.colab import files
import requests
from google.colab import drive
import csv

def load_checkpoint(path):
    data = {}
    if os.path.exists(path):
        with open(path, "r", newline='', encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                data[row["filename"]] = row["reason"]
    return data


def append_checkpoint(path, filename, reason):
    file_exists = os.path.exists(path)

    with open(path, "a", newline='', encoding="utf-8") as f:
        writer = csv.writer(f)

        if not file_exists:
            writer.writerow(["filename", "reason"])

        writer.writerow([filename, reason])


drive.mount('/content/drive')

CHECKPOINT_PATH = "/content/drive/MyDrive/stage2_checkpoint.csv"

def download_yandex_disk(url, output_path):
    api_url = "https://cloud-api.yandex.net/v1/disk/public/resources/download"
    r = requests.get(api_url, params={"public_key": url})
    download_url = r.json()["href"]

    with requests.get(download_url, stream=True) as r:
        with open(output_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)


def detect_language_from_audio(audio):
    audio = whisper.pad_or_trim(audio)
    mel = whisper.log_mel_spectrogram(audio).to(model.device)
    _, probs = model.detect_language(mel)
    lang = max(probs, key=probs.get)
    return lang, probs[lang]


YANDEX_URL = "YANDEX DISK LINK"
ZIP_PATH = "stage1_filtered.zip"
INPUT_DIR = "stage1"
OUTPUT_DIR = "stage2_final"

download_yandex_disk(YANDEX_URL, ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(INPUT_DIR)

os.makedirs(OUTPUT_DIR, exist_ok=True)

model = whisper.load_model("small")

NO_SPEECH_THRESHOLD = 0.6

reasons = {
    "no_segments": 0,
    "whisper_no_speech": 0,
    "low_confidence": 0,
    #"lang_text_ru": 0,
    #"lang_text_en": 0,
    "lang_audio_ru": 0,
    "lang_audio_en": 0,
    #"ru_text": 0,
    "error": 0
}

# HELPERS

#def count_russian_words(text):
    #words = re.findall(r'\b\w+\b', text.lower())
    #russian_word_count = 0

    #for word in words:
        #if any('а' <= c <= 'я' or c == 'ё' for c in word):
            #russian_word_count += 1

    #return russian_word_count

def check_stage2(path):
    try:
        y, sr = librosa.load(path, sr=16000)

        result = model.transcribe(
            y,
            fp16=False,
            verbose=False,
            task="transcribe",
            condition_on_previous_text=False
        )

        segments = result.get("segments", [])
        if not segments:
            return False, "no_segments"

        speech_segments = [
            s for s in segments
            if s["no_speech_prob"] < NO_SPEECH_THRESHOLD
        ]

        if not speech_segments:
            return False, "whisper_no_speech"

        if result.get("avg_logprob", -1) < -1.3:
            return False, "low_confidence"

        lang, prob = detect_language_from_audio(y)

        if lang in ["ru"] and prob > 0.7:
            return False, f"lang_audio_{lang}"

        #language = result.get("language", None)

        #if language in ["ru", "en"] and result.get("avg_logprob", -1) > -1.2:
            #return False, f"lang_text_{language}"

        #text = result.get("text", "")
        #if count_russian_words(text) >= 5:
            #return False, "ru_text"

        return True, "ok"

    except:
        return False, "error"

checkpoint = load_checkpoint(CHECKPOINT_PATH)

kept = 0
removed = 0

for fname, reason in checkpoint.items():
    if reason == "ok":
        kept += 1
    else:
        removed += 1
        if reason in reasons:
            reasons[reason] += 1

files_list = []
for root, _, fs in os.walk(INPUT_DIR):
    for f in fs:
        if f.lower().endswith(".wav"):
            files_list.append(os.path.join(root, f))

for path in files_list:
    fname = os.path.basename(path)

    if fname in checkpoint and checkpoint[fname] == "ok":
        dst = os.path.join(OUTPUT_DIR, fname)

        if not os.path.exists(dst):
            shutil.copy(path, dst)

for path in tqdm(files_list):
    fname = os.path.basename(path)

    if fname in checkpoint:
        continue

    ok, reason = check_stage2(path)

    append_checkpoint(CHECKPOINT_PATH, fname, reason)

    if ok:
        shutil.copy(path, os.path.join(OUTPUT_DIR, fname))
        kept += 1
    else:
        removed += 1
        reasons[reason] += 1

print("\nFINAL STATS")
print(f"Kept: {kept}")
print(f"Removed: {removed}")

print("\nREASONS")
for k, v in reasons.items():
    print(f"{k}: {v}")

# ZIP
shutil.make_archive("4_non-urmi_unlabeled", 'zip', OUTPUT_DIR)
files.download("4_non-urmi_unlabeled.zip")